In [1]:
import os
import json
import numpy as np
import nibabel as nib

In [2]:
with open("data_split.json", "r") as f:
    split_data = json.load(f)

train_subjects = split_data["train"]
val_subjects = split_data["validation"]
test_subjects = split_data["test"]

print("Train:", len(train_subjects))
print("Validation:", len(val_subjects))
print("Test:", len(test_subjects))

Train: 1000
Validation: 125
Test: 126


In [3]:
train_data = r"Data/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"

In [4]:
print(os.path.exists(train_data))
print(len(os.listdir(train_data)))

True
1251


In [5]:
def preprocess_t2f(image):
    # Expected original BraTS shape
    if image.shape != (240, 240, 155):
        raise ValueError(f"Unexpected image shape: {image.shape}")

    image = image[16:224, 8:232, :]

    image = np.pad(
        image,
        ((0, 0), (0, 0), (2, 3)),
        mode="constant",
        constant_values=0
    )

    foreground = image > 0

    if not np.any(foreground):
        raise ValueError("No foreground voxels found")

    upper = np.percentile(image[foreground], 99.9)

    image = np.clip(image, 0, upper)

    image = image / upper

    image[~foreground] = 0

    return image.astype(np.float32)

In [6]:
subject = train_subjects[0]
subject_path = os.path.join(train_data, subject)

files = os.listdir(subject_path)

t2f_file = [f for f in files if "t2f" in f.lower()][0]

t2f = nib.load(
    os.path.join(subject_path, t2f_file)
).get_fdata()

processed = preprocess_t2f(t2f)

print("Subject:", subject)
print("Original shape:", t2f.shape)
print("Processed shape:", processed.shape)
print("Min:", processed.min())
print("Max:", processed.max())
print("dtype:", processed.dtype)

Subject: BraTS-GLI-00240-000
Original shape: (240, 240, 155)
Processed shape: (208, 224, 160)
Min: 0.0
Max: 1.0
dtype: float32


In [7]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.13.0+cu132
CUDA available: True


In [8]:
from torch.utils.data import Dataset, DataLoader

class BraTSDataset(Dataset):
    def __init__(self, subjects, data_dir):
        self.subjects = subjects
        self.data_dir = data_dir

    def __len__(self):
        return len(self.subjects)

    def __getitem__(self, idx):
        subject = self.subjects[idx]
        subject_path = os.path.join(self.data_dir, subject)

        files = os.listdir(subject_path)

        t2f_file = [f for f in files if "t2f" in f.lower()][0]

        image = nib.load(
            os.path.join(subject_path, t2f_file)
        ).get_fdata()

        image = preprocess_t2f(image)

        image = torch.from_numpy(image).unsqueeze(0)

        return {
            "image": image,
            "subject": subject
        }

In [9]:
train_dataset = BraTSDataset(
    subjects=train_subjects,
    data_dir=train_data
)

print("Dataset size:", len(train_dataset))

Dataset size: 1000


In [10]:
sample = train_dataset[0]

print("Subject:", sample["subject"])
print("Image shape:", sample["image"].shape)
print("dtype:", sample["image"].dtype)
print("Min:", sample["image"].min().item())
print("Max:", sample["image"].max().item())

Subject: BraTS-GLI-00240-000
Image shape: torch.Size([1, 208, 224, 160])
dtype: torch.float32
Min: 0.0
Max: 1.0


In [11]:
train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0
)

batch = next(iter(train_loader))

print("Batch image shape:", batch["image"].shape)
print("Batch dtype:", batch["image"].dtype)
print("Subject:", batch["subject"])
print("Min:", batch["image"].min().item())
print("Max:", batch["image"].max().item())

Batch image shape: torch.Size([1, 1, 208, 224, 160])
Batch dtype: torch.float32
Subject: ['BraTS-GLI-00017-000']
Min: 0.0
Max: 1.0


In [12]:
import torch

timesteps = 1000

beta_start = 1e-4
beta_end = 0.02

betas = torch.linspace(beta_start, beta_end, timesteps)

alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)

sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)

print("Timesteps:", timesteps)
print("Beta shape:", betas.shape)
print("First beta:", betas[0].item())
print("Last beta:", betas[-1].item())

Timesteps: 1000
Beta shape: torch.Size([1000])
First beta: 9.999999747378752e-05
Last beta: 0.019999999552965164


In [13]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        device = t.device
        half_dim = self.dim // 2

        embeddings = math.log(10000) / (half_dim - 1)

        embeddings = torch.exp(
            torch.arange(half_dim, device=device) * -embeddings
        )

        embeddings = t[:, None] * embeddings[None, :]

        embeddings = torch.cat(
            (embeddings.sin(), embeddings.cos()),
            dim=1
        )

        return embeddings

In [14]:
time_embed = SinusoidalTimeEmbedding(128)

t = torch.tensor([0, 100, 500, 999])

emb = time_embed(t)

print("Input timesteps:", t.shape)
print("Embedding shape:", emb.shape)

Input timesteps: torch.Size([4])
Embedding shape: torch.Size([4, 128])


In [15]:
class ResBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim):
        super().__init__()

        self.conv1 = nn.Conv3d(
            in_channels, out_channels,
            kernel_size=3, padding=1
        )

        self.conv2 = nn.Conv3d(
            out_channels, out_channels,
            kernel_size=3, padding=1
        )

        self.norm1 = nn.GroupNorm(
            num_groups=8,
            num_channels=out_channels
        )

        self.norm2 = nn.GroupNorm(
            num_groups=8,
            num_channels=out_channels
        )

        self.time_mlp = nn.Linear(
            time_dim,
            out_channels
        )

        if in_channels != out_channels:
            self.residual = nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=1
            )
        else:
            self.residual = nn.Identity()

    def forward(self, x, t):
        h = self.conv1(x)
        h = self.norm1(h)
        h = F.silu(h)

        time_emb = self.time_mlp(t)
        time_emb = time_emb[:, :, None, None, None]

        h = h + time_emb

        h = self.conv2(h)
        h = self.norm2(h)
        h = F.silu(h)

        return h + self.residual(x)

In [16]:
block = ResBlock3D(
    in_channels=16,
    out_channels=32,
    time_dim=128
)

x_test = torch.randn(
    2, 16, 16, 16, 16
)

t_test = torch.randn(
    2, 128
)

y_test = block(x_test, t_test)

print("Input shape:", x_test.shape)
print("Output shape:", y_test.shape)

Input shape: torch.Size([2, 16, 16, 16, 16])
Output shape: torch.Size([2, 32, 16, 16, 16])


In [17]:
class DownBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim):
        super().__init__()

        self.resblock = ResBlock3D(
            in_channels,
            out_channels,
            time_dim
        )

        self.downsample = nn.Conv3d(
            out_channels,
            out_channels,
            kernel_size=4,
            stride=2,
            padding=1
        )

    def forward(self, x, t):
        h = self.resblock(x, t)

        down = self.downsample(h)

        return h, down


class UpBlock3D(nn.Module):
    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
        time_dim
    ):
        super().__init__()

        self.upsample = nn.ConvTranspose3d(
            in_channels,
            out_channels,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.resblock = ResBlock3D(
            out_channels + skip_channels,
            out_channels,
            time_dim
        )

    def forward(self, x, skip, t):
        x = self.upsample(x)

        x = torch.cat([x, skip], dim=1)

        x = self.resblock(x, t)

        return x

In [18]:
down_block = DownBlock3D(
    in_channels=16,
    out_channels=32,
    time_dim=128
)

x_test = torch.randn(2, 16, 16, 16, 16)
t_test = torch.randn(2, 128)

skip, down = down_block(x_test, t_test)

print("Original:", x_test.shape)
print("Skip:", skip.shape)
print("Downsampled:", down.shape)

Original: torch.Size([2, 16, 16, 16, 16])
Skip: torch.Size([2, 32, 16, 16, 16])
Downsampled: torch.Size([2, 32, 8, 8, 8])


In [19]:
up_block = UpBlock3D(
    in_channels=32,
    skip_channels=32,
    out_channels=32,
    time_dim=128
)

up = up_block(down, skip, t_test)

print("Upsampled:", up.shape)

Upsampled: torch.Size([2, 32, 16, 16, 16])


In [20]:
class UNet3D(nn.Module):
    def __init__(
        self,
        in_channels=1,
        out_channels=1,
        base_channels=16,
        time_dim=128
    ):
        super().__init__()

        self.time_embedding = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )

        self.input_conv = nn.Conv3d(
            in_channels,
            base_channels,
            kernel_size=3,
            padding=1
        )

        self.down1 = DownBlock3D(
            base_channels,
            base_channels * 2,
            time_dim
        )

        self.down2 = DownBlock3D(
            base_channels * 2,
            base_channels * 4,
            time_dim
        )

        self.down3 = DownBlock3D(
            base_channels * 4,
            base_channels * 8,
            time_dim
        )

        self.mid = ResBlock3D(
            base_channels * 8,
            base_channels * 8,
            time_dim
        )

        self.up3 = UpBlock3D(
            in_channels=base_channels * 8,
            skip_channels=base_channels * 8,
            out_channels=base_channels * 4,
            time_dim=time_dim
        )

        self.up2 = UpBlock3D(
            in_channels=base_channels * 4,
            skip_channels=base_channels * 4,
            out_channels=base_channels * 2,
            time_dim=time_dim
        )

        self.up1 = UpBlock3D(
            in_channels=base_channels * 2,
            skip_channels=base_channels * 2,
            out_channels=base_channels,
            time_dim=time_dim
        )

        self.output_conv = nn.Conv3d(
            base_channels,
            out_channels,
            kernel_size=1
        )

    def forward(self, x, t):
        t = self.time_embedding(t)

        x = self.input_conv(x)

        skip1, x = self.down1(x, t)
        skip2, x = self.down2(x, t)
        skip3, x = self.down3(x, t)

        x = self.mid(x, t)

        x = self.up3(x, skip3, t)
        x = self.up2(x, skip2, t)
        x = self.up1(x, skip1, t)

        x = self.output_conv(x)

        return x

In [21]:
model = UNet3D(
    in_channels=1,
    out_channels=1,
    base_channels=16,
    time_dim=128
)

x_test = torch.randn(
    1, 1, 64, 64, 64
)

t_test = torch.tensor([500])

with torch.no_grad():
    y_test = model(x_test, t_test)

print("Input shape:", x_test.shape)
print("Output shape:", y_test.shape)

Input shape: torch.Size([1, 1, 64, 64, 64])
Output shape: torch.Size([1, 1, 64, 64, 64])


In [22]:
def q_sample(x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)

    sqrt_alpha_hat = sqrt_alphas_cumprod[t].view(-1, 1, 1, 1, 1)
    sqrt_one_minus_alpha_hat = sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1, 1)

    xt = (
        sqrt_alpha_hat * x0
        + sqrt_one_minus_alpha_hat * noise
    )

    return xt, noise

In [23]:
batch = next(iter(train_loader))

x0 = batch["image"]

t = torch.randint(
    0,
    timesteps,
    (x0.shape[0],)
)

xt, noise = q_sample(x0, t)

print("Original shape:", x0.shape)
print("Noisy shape:", xt.shape)
print("Noise shape:", noise.shape)
print("Timestep:", t)

Original shape: torch.Size([1, 1, 208, 224, 160])
Noisy shape: torch.Size([1, 1, 208, 224, 160])
Noise shape: torch.Size([1, 1, 208, 224, 160])
Timestep: tensor([247])


In [24]:
model = UNet3D(
    in_channels=1,
    out_channels=1,
    base_channels=16,
    time_dim=128
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

x0 = batch["image"]

t = torch.randint(
    0,
    timesteps,
    (x0.shape[0],)
)

xt, noise = q_sample(x0, t)

predicted_noise = model(xt, t)

loss = F.mse_loss(
    predicted_noise,
    noise
)

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("Loss:", loss.item())

Loss: 1.152634620666504


In [39]:
def train_ddpm(
    model,
    train_loader,
    epochs,
    optimizer,
    device,
    checkpoint_dir="checkpoints"
):
    import os

    os.makedirs(checkpoint_dir, exist_ok=True)

    model.train()

    # Move diffusion schedule to the same device once
    sqrt_alpha_cumprod_device = sqrt_alphas_cumprod.to(device)
    sqrt_one_minus_alpha_cumprod_device = (
        sqrt_one_minus_alphas_cumprod.to(device)
    )

    for epoch in range(epochs):
        epoch_loss = 0.0

        for batch_idx, batch in enumerate(train_loader):
            x0 = batch["image"].to(device)

            t = torch.randint(
                0,
                timesteps,
                (x0.shape[0],),
                device=device
            )

            noise = torch.randn_like(x0)

            sqrt_alpha_hat = (
                sqrt_alpha_cumprod_device[t]
                .view(-1, 1, 1, 1, 1)
            )

            sqrt_one_minus_alpha_hat = (
                sqrt_one_minus_alpha_cumprod_device[t]
                .view(-1, 1, 1, 1, 1)
            )

            xt = (
                sqrt_alpha_hat * x0
                + sqrt_one_minus_alpha_hat * noise
            )

            predicted_noise = model(xt, t)

            loss = F.mse_loss(
                predicted_noise,
                noise
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

            if (batch_idx + 1) % 10 == 0:
                print(
                    f"Epoch {epoch + 1}/{epochs} | "
                    f"Batch {batch_idx + 1}/{len(train_loader)} | "
                    f"Loss: {loss.item():.4f}"
                )

        avg_loss = epoch_loss / len(train_loader)

        print(
            f"Epoch {epoch + 1} completed | "
            f"Average loss: {avg_loss:.4f}"
        )

        # Save checkpoint after every epoch
        checkpoint_path = os.path.join(
            checkpoint_dir,
            f"ddpm_epoch_{epoch + 1:03d}.pt"
        )

        save_checkpoint(
            model=model,
            optimizer=optimizer,
            epoch=epoch + 1,
            path=checkpoint_path
        )

        print("Saved:", checkpoint_path)

In [26]:
from torch.utils.data import Subset

small_dataset = Subset(
    train_dataset,
    range(2)
)

small_loader = DataLoader(
    small_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0
)

device = torch.device("cpu")

model = UNet3D(
    in_channels=1,
    out_channels=1,
    base_channels=16,
    time_dim=128
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

train_ddpm(
    model=model,
    train_loader=small_loader,
    epochs=1,
    optimizer=optimizer,
    device=device
)

Epoch 1 completed | Average loss: 1.1025


In [29]:
def save_checkpoint(model, optimizer, epoch, path):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict()
    }, path)


def load_checkpoint(model, optimizer, path, device):
    checkpoint = torch.load(path, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])

    if optimizer is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    return checkpoint["epoch"]

In [30]:
@torch.no_grad()
def sample_ddpm(model, shape, device, sample_timesteps=None):
    model.eval()

    if sample_timesteps is None:
        sample_timesteps = timesteps

    x = torch.randn(shape, device=device)

    for t in reversed(range(sample_timesteps)):
        t_batch = torch.full(
            (shape[0],),
            t,
            device=device,
            dtype=torch.long
        )

        beta_t = betas[t].to(device)
        alpha_t = alphas[t].to(device)
        alpha_hat_t = alphas_cumprod[t].to(device)

        predicted_noise = model(x, t_batch)

        model_mean = (
            1 / torch.sqrt(alpha_t)
        ) * (
            x
            - (
                beta_t
                / torch.sqrt(1 - alpha_hat_t)
            ) * predicted_noise
        )

        if t > 0:
            noise = torch.randn_like(x)
            x = model_mean + torch.sqrt(beta_t) * noise
        else:
            x = model_mean

    return x

In [31]:
device = torch.device("cpu")

test_sample = sample_ddpm(
    model=model,
    shape=(1, 1, 32, 32, 32),
    device=device,
    sample_timesteps=10
)

print("Generated shape:", test_sample.shape)
print("Min:", test_sample.min().item())
print("Max:", test_sample.max().item())
print("NaN:", torch.isnan(test_sample).any().item())

Generated shape: torch.Size([1, 1, 32, 32, 32])
Min: -3.7871358394622803
Max: 3.9831717014312744
NaN: False


In [32]:
checkpoint_path = "ddpm_checkpoint.pt"

save_checkpoint(
    model=model,
    optimizer=optimizer,
    epoch=1,
    path=checkpoint_path
)

print("Checkpoint saved:", checkpoint_path)

Checkpoint saved: ddpm_checkpoint.pt


In [33]:
import sys
import torch

print("Pythin:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPT:", torch.cuda.get_device_name(0))

Pythin: /users/tdd540/.local/share/flight/env/conda+python_env/bin/python
PyTorch: 2.13.0+cu132
CUDA available: True
GPT: NVIDIA A40


In [34]:
%pip install numpy nibabel

Note: you may need to restart the kernel to use updated packages.


In [35]:
import numpy as np
import nibabel as nib

print("NumPy:", np.__version__)
print("Nibabel:", nib.__version__)

NumPy: 2.5.2
Nibabel: 5.4.2


In [36]:
import os

train_data = r"Data/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"

print(os.path.exists(train_data))
print(len(os.listdir(train_data)))

True
1251


In [38]:
device = torch.device("cuda")

model = UNet3D(
    in_channels=1,
    out_channels=1,
    base_channels=16,
    time_dim=128
).to(device)

batch = next(iter(train_loader))
x0 = batch["image"].to(device)

# 把 diffusion schedule 搬到 GPU
sqrt_alphas_cumprod_gpu = sqrt_alphas_cumprod.to(device)
sqrt_one_alphas_cumprod_gpu = sqrt_one_minus_alphas_cumprod.to(device)

t = torch.randint(
    0,
    timesteps,
    (x0.shape[0],),
    device=device
)

noise = torch.randn_like(x0)

sqrt_alpha_hat = sqrt_alphas_cumprod_gpu[t].view(
    -1, 1, 1, 1, 1
)

sqrt_one_minus_alpha_hat = sqrt_one_alphas_cumprod_gpu[t].view(
    -1, 1, 1, 1, 1
)

xt = (
    sqrt_alpha_hat * x0
    + sqrt_one_minus_alpha_hat * noise
)

predicted_noise = model(xt, t)

loss = F.mse_loss(
    predicted_noise,
    noise
)

model.zero_grad()
loss.backward()

print("Input shape:", x0.shape)
print("Predicted shape:", predicted_noise.shape)
print("Loss:", loss.item())
print("GPU:", torch.cuda.get_device_name(0))
print(
    "Allocated GPU memory (GB):",
    torch.cuda.memory_allocated() / 1024**3
)

Input shape: torch.Size([1, 1, 208, 224, 160])
Predicted shape: torch.Size([1, 1, 208, 224, 160])
Loss: 1.2204411029815674
GPU: NVIDIA A40
Allocated GPU memory (GB): 0.19024419784545898


In [40]:
from torch.utils.data import Subset

small_dataset = Subset(
    train_dataset,
    range(2)
)

small_loader = DataLoader(
    small_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0
)

device = torch.device("cuda")

model = UNet3D(
    in_channels=1,
    out_channels=1,
    base_channels=16,
    time_dim=128
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

train_ddpm(
    model=model,
    train_loader=small_loader,
    epochs=1,
    optimizer=optimizer,
    device=device,
    checkpoint_dir="checkpoints_test"
)

Epoch 1 completed | Average loss: 1.1579
Saved: checkpoints_test/ddpm_epoch_001.pt


In [41]:
train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0
)

In [42]:
device = torch.device("cuda")

model = UNet3D(
    in_channels=1,
    out_channels=1,
    base_channels=16,
    time_dim=128
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [ ]:
train_ddpm(
    model=model,
    train_loader=train_loader,
    epochs=10,
    optimizer=optimizer,
    device=device,
    checkpoint_dir="checkpoints"
)